# Prédiction de la Réussite Académique des Étudiants
## Implémentation From Scratch et Analyse Mathématique en NumPy

**Auteurs & Cadre :** Projet individuel de Mathématiques Appliquées et Machine Learning.
**Objectif Pédagogique :** Développer from scratch (vectorisation matricielle avec NumPy) les algorithmes d'Analyse en Composantes Principales (ACP) et de Régression Logistique avec régularisation L2, et analyser leurs propriétés théoriques et empiriques.

## 2. Problématique Métier
*Dans quelle mesure les caractéristiques académiques et personnelles d'un étudiant permettent-elles de prédire sa réussite académique ?*

La réussite académique est modélisée par une variable binaire :
- $y = 1$ : Réussite académique (Note finale $G3 \ge 10/20$).
- $y = 0$ : Non-réussite académique (Note finale $G3 < 10/20$).

## 3. Dataset & Exploration des Données
Chargement du jeu de données (données synthétiques basées sur les distributions du dataset UCI Student Performance).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/student_data.csv')
print('Taille du dataset :', df.shape)
df.head()

## 4. Prétraitement et Standardisation From Scratch

**Formule mathématique :**
$$X_{standard} = \frac{X - \mu}{\sigma}$$

La standardisation est cruciale pour l'ACP (sensible aux échelles) et la descente de gradient (conditionnement du problème).

In [ ]:
from src.preprocessing import StandardScalerScratch, train_test_split_scratch

X = df.drop(columns=['academic_success']).values
y = df['academic_success'].values

X_train, X_test, y_train, y_test = train_test_split_scratch(X, y, test_size=0.2, random_state=42)

scaler = StandardScalerScratch()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Moyennes après scaling :', np.round(X_train_scaled.mean(axis=0), 2))
print('Écarts-types après scaling :', np.round(X_train_scaled.std(axis=0), 2))

## 5. Analyse en Composantes Principales (ACP) From Scratch

**Étape 1 : Matrice de covariance empirique :**
$$\Sigma = \frac{1}{m} X^T X$$

**Étape 2 : Décomposition spectrale :**
$$\Sigma v_i = \lambda_i v_i$$

**Étape 3 : Projection 2D :**
$$Z = X W$$

In [ ]:
from src.pca_scratch import PCAFromScratch

pca = PCAFromScratch(n_components=2)
Z_train = pca.fit_transform(X_train_scaled)

print('Ratio de variance expliquée (PC1, PC2) :', pca.explained_variance_ratio_)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(Z_train[:, 0], Z_train[:, 1], c=y_train, cmap='coolwarm', alpha=0.8, edgecolors='k')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Projection ACP 2D')
plt.colorbar(scatter, label='Réussite Académique')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 6. Régression Logistique & Descente de Gradient From Scratch

**Hypothèse :** $\hat{y} = \sigma(X \theta) = \frac{1}{1 + e^{-X \theta}}$

**Fonction de coût Log-Loss + L2 :**
$$J(\theta) = -\frac{1}{m} \sum [ y \log(\hat{y}) + (1-y)\log(1-\hat{y}) ] + \frac{\lambda}{2m} \sum_{j=1}^n \theta_j^2$$

**Gradient Matriciel Analytique :**
$$\nabla J(\theta) = \frac{1}{m} X^T (\hat{y} - y) + \frac{\lambda}{m} \theta$$

In [ ]:
from src.logistic_regression_scratch import LogisticRegressionScratch

model = LogisticRegressionScratch(learning_rate=0.1, l2_lambda=0.1, n_iterations=1000)
model.fit(X_train_scaled, y_train)

plt.figure(figsize=(7, 4))
plt.plot(model.cost_history, color='blue', linewidth=2)
plt.title('Courbe de Convergence de la Loss J(theta)')
plt.xlabel('Itérations')
plt.ylabel('Coût Log-Loss')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 7. Évaluation et Comparaison avec Scikit-Learn
Calcul des métriques et comparaison rigoureuse.

In [ ]:
from src.metrics import accuracy_score_scratch, f1_score_scratch, confusion_matrix_scratch

preds = model.predict(X_test_scaled)
print('Accuracy Test :', accuracy_score_scratch(y_test, preds))
print('F1-Score Test :', f1_score_scratch(y_test, preds))
print('Matrice de confusion :\n', confusion_matrix_scratch(y_test, preds))

## 8. Conclusion Scientifique
Les implémentations vectorisées en NumPy de l'ACP et de la Régression Logistique atteignent des performances identiques à Scikit-Learn tout en offrant une transparence mathématique totale.